## Get Physical Properties from AlphaFold Output

May need `openmmtools` and `openff-toolkit` from `conda-forge` and `omnia`

In [ ]:
%conda config --add channels conda-forge
%conda config --add channels omnia

# Install packages
%pip install -q numpy pandas matplotlib seaborn pathos biopython tdqm mdtraj rdkit MDAnalysis prolif openmm
%conda install openmmtools openff-toolkit

## Setup

In [ ]:
import prolif as plf
import MDAnalysis as mda
from MDAnalysis.analysis import contacts
from rdkit import Chem
from rdkit.Chem import AllChem
import os
import numpy as np

## MDAnalysis

In [ ]:
import MDAnalysis as mda
from MDAnalysis.lib.distances import distance_array
import numpy as np

# Load your PDB file
pdb_file = "my_complex.pdb" # <-- Your PDB file here

try:
    u = mda.Universe(pdb_file)
except Exception as e:
    print(f"Error loading {pdb_file}: {e}")
    exit()

# --- 1. Define Your Specific Parts ---
sel_A = "chainID A"
sel_B = "chainID B"

group_A = u.select_atoms(sel_A)
group_B = u.select_atoms(sel_B)

if group_A.n_atoms == 0 or group_B.n_atoms == 0:
    print("Error: One or both selections resulted in 0 atoms.")
    print(f"Check if '{sel_A}' and '{sel_B}' are correct for your PDB.")
    exit()

print(f"Group A ({sel_A}): {group_A.n_atoms} atoms")
print(f"Group B ({sel_B}): {group_B.n_atoms} atoms")

# --- 2. Run Analysis (New Method) ---
distance_cutoff = 4.5

# Calculate the distance matrix between all atoms in group A
# and all atoms in group B.
# This creates a (n_atoms_A, n_atoms_B) array.
dist_matrix = distance_array(group_A.positions, group_B.positions)

# Find the indices (i, j) where the distance is less than the cutoff
# i = index in group_A, j = index in group_B
close_atom_indices = np.where(dist_matrix <= distance_cutoff)

# 'close_atom_indices' is a tuple of two arrays:
# (array_of_i_indices, array_of_j_indices)

# --- 3. Map Atom Indices to Residues ---

# Get the indices of the atoms in group_A that are close
close_atoms_A_indices = close_atom_indices[0]
# Get the indices of the atoms in group_B that are close
close_atoms_B_indices = close_atom_indices[1]

# Now, map these atom indices to their parent residues
# We use a set to store the residues so we only get unique ones
interacting_residues_A = set()
for atom_index in close_atoms_A_indices:
    interacting_residues_A.add(group_A[atom_index].residue)

interacting_residues_B = set()
for atom_index in close_atoms_B_indices:
    interacting_residues_B.add(group_B[atom_index].residue)

# --- 4. Print Results ---
print("\n--- MDAnalysis Contact Analysis (within 4.5 Å) ---")

print(f"\nResidues in Chain A interacting with Chain B:")
if interacting_residues_A:
    # Sort the residues by resid for a clean output
    sorted_residues = sorted(list(interacting_residues_A), key=lambda r: r.resid)
    print([res.resname + str(res.resid) for res in sorted_residues])
else:
    print("None found.")

print(f"\nResidues in Chain B interacting with Chain A:")
if interacting_residues_B:
    sorted_residues = sorted(list(interacting_residues_B), key=lambda r: r.resid)
    print([res.resname + str(res.resid) for res in sorted_residues])
else:
    print("None found.")

# --- Optional: Hydrogen Bond Analysis ---
# (This still has the same limitation: it needs hydrogens in the PDB)
try:
    from MDAnalysis.analysis.hydrogenbonds import HydrogenBondAnalysis
    
    # Select all protein atoms for H-bond analysis
    # We select 'protein' and 'protein' to find intra-protein H-bonds
    h = HydrogenBondAnalysis(u, sel1="protein", sel2="protein")
    h.run()
    
    # Filter for H-bonds specifically between Chain A and Chain B
    inter_chain_hbonds = []
    for bond in h.results.hbonds:
        donor_chain = u.atoms[bond[2]].chainID
        acceptor_chain = u.atoms[bond[0]].chainID
        
        if (donor_chain == 'A' and acceptor_chain == 'B') or \
           (donor_chain == 'B' and acceptor_chain == 'A'):
            inter_chain_hbonds.append(bond)

    print(f"\nFound {len(inter_chain_hbonds)} total inter-chain H-bonds (A-B).")
    # Note: This will likely be 0 if your PDB has no hydrogens.

except ImportError:
    print("\nHydrogen bond analysis requires an updated version of MDAnalysis.")
except Exception as e:
    print(f"\nCould not run H-bond analysis: {e}")